In [1]:
!pip install sentence-transformers numpy

In [2]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 41.9 MB/s eta 0:00:00:00:0100:01


In [3]:
# Update the core libraries
!pip install -U transformers sentence-transformers einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 79.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3

In [4]:
pip install -U transformers huggingface_hub

Note: you may need to restart the kernel to use updated packages.


### Setting Up some configuration for the project

In [20]:
import os
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# Example: r"D:\your_project\data"
DATA_PATH = "/kaggle/input/datasets/ahmedfayad/data-markdown/data"

# Examples:
# "intfloat/multilingual-e5-large"
# "BAAI/bge-m3"
EMBEDDING_MODEL1 = "intfloat/multilingual-e5-large"
EMBEDDING_MODEL2 = "BAAI/bge-m3"
EMBEDDING_MODEL3 = "sentence-transformers/all-MiniLM-L6-v2"


### Provide Functions to make Chunking

In [21]:
# ============================================================
# 1. CHUNKING
# ============================================================

def chunk_text(text: str , val : int):
    """
    Splits a document into smaller chunks.

    Your task:
    - Split text into paragraphs
    - Merge them into chunks (~500–700 characters)

    Hint:
    Use regex to split by empty lines.
    """

    # Split into paragraphs
    paragraphs = re.split(r"\n\s*\n", text.strip())

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        # Clean paragraph (strip whitespace)
        para = para.strip()
        if not para:
            continue

        # Build chunks with size limit
        if len(current_chunk) + len(para) > val:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = para
        else:
            if current_chunk:
                current_chunk += "\n\n" + para
            else:
                current_chunk = para

    # Don't forget the last chunk
    if current_chunk:
        chunks.append(current_chunk)

    return chunks

### Writing a function to make embeddings for the chunks

In [25]:
# ============================================================
# 2. EMBEDDINGS
# ============================================================

def create_embeddings(chunks,EMBEDDING_MODEL):
    """
    Converts text chunks into vectors.

    FIX: normalize_embeddings is a parameter of model.encode(),
         NOT of SentenceTransformer() constructor.
    """

    print("Creating embeddings...")

    # Load model (no normalize_embeddings here)
    model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

    # Encode chunks — normalize_embeddings goes HERE
    embeddings = model.encode(
        chunks,
        show_progress_bar=True,
        normalize_embeddings=True   # ✅ correct place
    )

    # Convert to float32 (required by FAISS)
    embeddings = embeddings.astype(np.float32)

    return model, embeddings


### set up faiss to make the search for the relevant chunks faster

In [26]:
# ============================================================
# 3. FAISS INDEX
# ============================================================

def build_faiss_index(embeddings):
    """
    Builds a FAISS index for similarity search.

    IndexFlatIP = Inner Product (dot product).
    With normalized embeddings this equals cosine similarity.
    """

    dimension = embeddings.shape[1]

    # Create index
    index = faiss.IndexFlatIP(dimension)

    # Add embeddings to index
    index.add(embeddings)

    print(f"Index built with {index.ntotal} vectors of dimension {dimension}")

    return index


### Create Function to Retrieve the relevant chunks for the question

In [27]:
# ============================================================
# 4. RETRIEVAL
# ============================================================

def retrieve(query, model, index, chunks, metadata, top_k):
    """
    Finds the most relevant chunks.

    Steps:
    1. Encode query (with same normalization as chunks)
    2. Search FAISS
    3. Return results
    """

    print(f"\nSearching for: {query}")

    # Encode query — must use normalize_embeddings=True to match chunk embeddings
    query_emb = model.encode(
        [query],
        show_progress_bar=False,
        normalize_embeddings=True   # ✅ must match how chunks were encoded
    ).astype(np.float32)

    # Search index
    distances, indices = index.search(query_emb, top_k)

    results = []

    # Build results list
    for idx, score in zip(indices[0], distances[0]):
        if idx < len(chunks):
            results.append({
                "text": chunks[idx],
                "source": metadata[idx]["source"],
                "score": float(score)
            })

    return results

## Run main Pipeline to test different models and different chunks sizes

### Try first Embedding model `intfloat` chunk size --> 700

In [29]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()
            #chunk with 700
            doc_chunks = chunk_text(text,700)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL1)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 55 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 55 vectors of dimension 1024

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.8382):
FAQ: Why is my fiber internet slow?

أسئلة شائعة - بطء السرعة في الـ FTTH

سؤال شائع: "ليه النت بطيء أوي بعد تفعيل الفايبر؟"

الرد الموصى به:
"يا فندم، في أسباب كتير ممكن تسبب بطء في الـ FTTH:
1. WiFi
Source: faq_fiber_slow_speed.md

Result 2 (score: 0.8381):
لو المشكلة مستمرة بعد 30 دقيقة من الـ troubleshooting:
- لو العميل VIP → ارفع التذكرة كـ P1 فوراً.
- اعمل escalate للـ NOC team ودوّن كل الخطوات اللي عملتها.

نصيحة: قول للعميل "معلش يا فندم، هنحلها إ
Source: 5g_throttling_troubleshooting.md

Result 3 (score: 0.8353):
5G Throttling Troubleshooting Guide

دليل حل مشاكل الـ 5G Throttling

يا فندم، لو العميل شاكي إن السرعة بطيئة على الـ 5G، نتبع الخطوات دي بالترتيب:

أول حاجة نسأله: "هل تجاوزت الـ monthly quota بتاعتك
Source: 5g_throttling_troubleshooting.md

Result 4 (score: 0.8307):
الخطوات كاملة:
1. الـ agent يفتح الـ CRM ويأكد من الـ contract ID ويطلب الـ ONT seri

### Try first Embedding model `intfloat` chunk size --> 500

In [30]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()
            #chunk with 700
            doc_chunks = chunk_text(text,500)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL1)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 72 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Index built with 72 vectors of dimension 1024

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.8428):
FAQ: Why is my fiber internet slow?

أسئلة شائعة - بطء السرعة في الـ FTTH

سؤال شائع: "ليه النت بطيء أوي بعد تفعيل الفايبر؟"

الرد الموصى به:
"يا فندم، في أسباب كتير ممكن تسبب بطء في الـ FTTH:
1. WiFi
Source: faq_fiber_slow_speed.md

Result 2 (score: 0.8416):
نصيحة للـ support agents:
لو العميل شاكي من بطء في الذروة، قوله:
"يا فندم، ده normal في peak hours بسبب الاستخدام العالي. السرعة هترجع طبيعي بعد 11 مساءً. لو عايز حل دائم نقدر نراجع باقتك."

تابع الـ 
Source: peak_hour_management.md

Result 3 (score: 0.8367):
المشاكل الشائعة وحلولها:
- App not loading: Clear cache + update to latest version from Google Play / App Store.
- Can't see current usage: Check internet connection أو force logout and login again.
-
Source: nile_tel_mobile_app_troubleshooting.md

Result 4 (score: 0.8356):
5G Throttling Troubleshooting Guide

دليل حل مشاكل الـ 5G Throttling

يا فندم، لو العمي

### Try first Embedding model `intfloat` chunk size --> 600

In [31]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()
            #chunk with 700
            doc_chunks = chunk_text(text,600)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL1)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 61 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 61 vectors of dimension 1024

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.8382):
FAQ: Why is my fiber internet slow?

أسئلة شائعة - بطء السرعة في الـ FTTH

سؤال شائع: "ليه النت بطيء أوي بعد تفعيل الفايبر؟"

الرد الموصى به:
"يا فندم، في أسباب كتير ممكن تسبب بطء في الـ FTTH:
1. WiFi
Source: faq_fiber_slow_speed.md

Result 2 (score: 0.8356):
5G Throttling Troubleshooting Guide

دليل حل مشاكل الـ 5G Throttling

يا فندم، لو العميل شاكي إن السرعة بطيئة على الـ 5G، نتبع الخطوات دي بالترتيب:

أول حاجة نسأله: "هل تجاوزت الـ monthly quota بتاعتك
Source: 5g_throttling_troubleshooting.md

Result 3 (score: 0.8298):
الخطوات كاملة:
1. الـ agent يفتح الـ CRM ويأكد من الـ contract ID ويطلب الـ ONT serial number.
2. يشيك في الـ OSS system على توافر الـ splitter port في العنوان.
3. يعمل work order ويبعت الـ field engi
Source: fiber_activation_policy_2026.md

Result 4 (score: 0.8291):
في أوقات الذروة (من 8 لـ 11 مساءً) الـ congestion بيكون عالي جداً خاصة في المنصورة وم

### Try second model `BAAI` --> 700 chunk size

In [32]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()

            doc_chunks = chunk_text(text,700)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL2)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 55 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 55 vectors of dimension 1024

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.6175):
5G Throttling Troubleshooting Guide

دليل حل مشاكل الـ 5G Throttling

يا فندم، لو العميل شاكي إن السرعة بطيئة على الـ 5G، نتبع الخطوات دي بالترتيب:

أول حاجة نسأله: "هل تجاوزت الـ monthly quota بتاعتك
Source: 5g_throttling_troubleshooting.md

Result 2 (score: 0.6108):
FAQ: Why is my fiber internet slow?

أسئلة شائعة - بطء السرعة في الـ FTTH

سؤال شائع: "ليه النت بطيء أوي بعد تفعيل الفايبر؟"

الرد الموصى به:
"يا فندم، في أسباب كتير ممكن تسبب بطء في الـ FTTH:
1. WiFi
Source: faq_fiber_slow_speed.md

Result 3 (score: 0.6070):
لو المشكلة مستمرة بعد 30 دقيقة من الـ troubleshooting:
- لو العميل VIP → ارفع التذكرة كـ P1 فوراً.
- اعمل escalate للـ NOC team ودوّن كل الخطوات اللي عملتها.

نصيحة: قول للعميل "معلش يا فندم، هنحلها إ
Source: 5g_throttling_troubleshooting.md

Result 4 (score: 0.5745):
Network Outage Communication Templates

قوالب الرد على انقطاع الشبكة

يا فندم، لما ي

### Try second model `BAAI` --> 600 chunk size

In [33]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()

            doc_chunks = chunk_text(text,600)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL2)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 61 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 61 vectors of dimension 1024

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.6124):
5G Throttling Troubleshooting Guide

دليل حل مشاكل الـ 5G Throttling

يا فندم، لو العميل شاكي إن السرعة بطيئة على الـ 5G، نتبع الخطوات دي بالترتيب:

أول حاجة نسأله: "هل تجاوزت الـ monthly quota بتاعتك
Source: 5g_throttling_troubleshooting.md

Result 2 (score: 0.6108):
FAQ: Why is my fiber internet slow?

أسئلة شائعة - بطء السرعة في الـ FTTH

سؤال شائع: "ليه النت بطيء أوي بعد تفعيل الفايبر؟"

الرد الموصى به:
"يا فندم، في أسباب كتير ممكن تسبب بطء في الـ FTTH:
1. WiFi
Source: faq_fiber_slow_speed.md

Result 3 (score: 0.5669):
في أوقات الذروة (من 8 لـ 11 مساءً) الـ congestion بيكون عالي جداً خاصة في المنصورة ومناطق الدلتا، فكتير من الشكاوى بتيجي في رمضان بسبب الفيديوهات والستريمينج.

لو المشكلة مستمرة بعد 30 دقيقة من الـ tr
Source: 5g_throttling_troubleshooting.md

Result 4 (score: 0.5563):
Network Outage Communication Templates

قوالب الرد على انقطاع الشبكة

يا فندم، لما ي

### Try Sentence Mini Transformer Model --> 700 chunk size

In [34]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()

            doc_chunks = chunk_text(text,700)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL3)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 55 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 55 vectors of dimension 384

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.4385):
بعد ما ترجع الخدمة:
- أرسل follow-up message: "الخدمة رجعت يا فندم، لو لسه في مشكلة ابعتلنا فوراً."

Last Updated: March 2026 | Department: NOC
Source: network_outage_template.md

Result 2 (score: 0.3446):
نصيحة: دايماً قول "يا فندم، إحنا ملتزمين بكل لوائح الـ NTRA وهنحل مشكلتك في أسرع وقت".

Last Updated: March 2026 | Department: Legal & Compliance
Source: ntra_regulations_summary.md

Result 3 (score: 0.3420):
نصيحة للـ agent: استخدم كلام لطيف زي "يا فندم، أنا آسف جداً على المشكلة دي، وهنحلها في أسرع وقت ونبلغك بالـ compensation لو لزم الأمر".

Last Updated: February 2026 | Department: Customer Experience
Source: sla_vip_golden_customers.md

Result 4 (score: 0.3329):
نصيحة للـ agent: "يا فندم، خلينا نحاول نحل المشكلة اللي مخليك تلغي قبل ما نكمل الإجراءات."

Last Updated: April 2026 | Department: Retention & Billing
Source: contract_cancellation_process.md

Result 5 (sco

### Try Sentence Mini Transformer Model --> 600 chunk size

## Comparsion betweeb 3 models in response 

In [35]:
# ============================================================
# 5. MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    all_chunks = []
    metadata = []

    # -------------------------
    # Load data
    # -------------------------
    for file in os.listdir(DATA_PATH):
        if file.endswith(".md"):
            with open(os.path.join(DATA_PATH, file), "r", encoding="utf-8") as f:
                text = f.read()

            doc_chunks = chunk_text(text,600)

            for chunk in doc_chunks:
                all_chunks.append(chunk)
                metadata.append({"source": file})

    print(f"Loaded {len(all_chunks)} chunks from {DATA_PATH}")

    # -------------------------
    # Embeddings
    # -------------------------
    model, embeddings = create_embeddings(all_chunks,EMBEDDING_MODEL3)

    # -------------------------
    # Index
    # -------------------------
    index = build_faiss_index(embeddings)

    # -------------------------
    # Test
    # -------------------------
    query = "حل مشكلة بطء شبكة اﻻنترنت ؟؟"

    results = retrieve(
        query=query,
        model=model,
        index=index,
        chunks=all_chunks,
        metadata=metadata,
        top_k=6
    )

    # -------------------------
    # Output
    # -------------------------
    for i, r in enumerate(results):
        print(f"\nResult {i+1} (score: {r['score']:.4f}):")
        print(r["text"][:200])
        print("Source:", r["source"])

Loaded 61 chunks from /kaggle/input/datasets/ahmedfayad/data-markdown/data
Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Index built with 61 vectors of dimension 384

Searching for: حل مشكلة بطء شبكة اﻻنترنت ؟؟

Result 1 (score: 0.5149):
نصيحة: لو في شك، ارفعها P2 أحسن من P3. 
دايماً أضف reason للـ priority في description التذكرة.

Last Updated: February 2026 | Operations Department
Source: ticket_priority_guidelines.md

Result 2 (score: 0.4852):
لو العميل شاكي من رسوم عالية:
- افتح الـ usage logs في الـ BSS.
- لو في خطأ، اعمل credit note.
- لو مش خطأ، شرح له الـ charges بالتفصيل.

Last Updated: February 2026 | Customer Support
Source: roaming_charges_guide.md

Result 3 (score: 0.4589):
نصيحة: لو المشكلة فيها أكتر من فئة، اختار الأهم ودوّن التفاصيل في الـ description.

Last Updated: March 2026 | Operations
Source: ticket_categories_list.md

Result 4 (score: 0.3909):
نصيحة: "يا فندم، لو الـ ONT فيه عطل، هنبعتلك واحد جديد مجاناً ونركبه في أسرع وقت."

Last Updated: April 2026 | Technical Support
Source: common_ont_hardware_faults.md

Result 5 (score: 0.3761):
تابع الـ KPI dashboard يومياً عشان تشوف مناطق ال

### we can see that first model --> `Multilingul e5 large` reach the meaning of the query but it biased toword spacific keywords


### the second one is BGE - m3 which I conisder it the best one due to understanding query and try to fetch extact solution that can answer query 

### Thr 3rd one which is the lowest score response that give answers about solution dirctly and has no fully understanding to words

# Embedding Models Comparison Report

This report evaluates the performance of three different embedding models on an Arabic retrieval query (`"حل مشكلة بطء شبكة اﻻنترنت ؟؟"`).

## 1. Multilingual E5 Large (`intfloat/multilingual-e5-large`)
* **Performance:** Moderate / Biased
* **Observation:** The model successfully grasps the general meaning of the query. However, it exhibits a bias towards specific keywords rather than fully capturing the semantic context of the problem.

## 2. BGE-M3 (`BAAI/bge-m3`)
* **Performance:** **Best Overall** 🏆
* **Observation:** This model demonstrates the deepest semantic understanding of the query. Instead of just keyword matching, it effectively interprets the user's intent and attempts to fetch the exact solution that directly answers the query.

## 3. All-MiniLM-L6-v2 (`sentence-transformers/all-MiniLM-L6-v2`)
* **Performance:** Lowest Score
* **Observation:** This model struggles with full word comprehension and semantic understanding for this specific query. It tends to retrieve direct solutions but lacks the deeper contextual understanding shown by the larger multilingual models.

## The Effect of Chunk Size Tuning
During the experiments, text chunks were tuned to sizes of **500, 600, and 700 characters** to examine the impact on retrieval quality:
* **Smaller Chunks (500):** Yielded higher precision for dense, specific keyword matches but posed a risk of *context fragmentation* (e.g., separating a problem statement from its corresponding solution).
* **Larger Chunks (700):** Maintained excellent continuity and provided complete context, which is essential for detailed troubleshooting guides, though it slightly diluted the similarity scoring due to extra text.
* **Balanced Chunks (600):** Proved to be an effective sweet spot, maintaining enough context to house both the user's issue and the actionable steps without losing focus or score accuracy.

### Conclusion
For non-English or specialized queries, **`BAAI/bge-m3`** provides the most accurate and context-aware retrieval. When paired with an optimized chunk size (around **600-700 characters**) to preserve troubleshooting context, it stands as the most robust choice among the tested configurations for this RAG pipeline.